# Platform setup

Create the phase-1 market/macro schemas and Delta tables in Unity Catalog.

The catalog is expected to exist already. By default this notebook targets `market_macro`.

This notebook is create-only: it creates missing schemas and tables, then adds missing Bronze constraints.

In [ ]:
dbutils.widgets.text("catalog", "market_macro")

catalog = dbutils.widgets.get("catalog").strip() or "market_macro"

schemas = [
    "brz_market",
    "slv_market",
    "gld_market",
    "brz_macro",
    "slv_macro",
    "gld_macro",
    "gld_cross",
    "obs",
]

def constraint_exists(table_name: str, constraint_name: str) -> bool:
    property_key = f"delta.constraints.{constraint_name}"
    properties = spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
    return any(row.key == property_key for row in properties)

bronze_table_specs = {
    "brz_market.raw_coinbase_ohlc_1d": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_market.raw_coinbase_ohlc_1d (
            product_id STRING NOT NULL COMMENT 'Trading pair like BTC-USD',
            bar_date DATE NOT NULL COMMENT 'UTC date of the OHLC bar',
            open DOUBLE,
            high DOUBLE,
            low DOUBLE,
            close DOUBLE,
            volume DOUBLE,
            source_window_start TIMESTAMP COMMENT 'Requested window start used by ingestion (UTC)',
            source_window_end TIMESTAMP COMMENT 'Requested window end used by ingestion (UTC)',
            ingested_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp (UTC)',
            run_id STRING NOT NULL COMMENT 'Pipeline run identifier',
            payload_hash STRING COMMENT 'Hash of raw API payload for audit/dedup'
        )
        USING DELTA
        COMMENT 'Bronze: raw daily OHLC bars ingested from Coinbase. Logical PK = (product_id, bar_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_coinbase_ohlc_keys": """
            ALTER TABLE `{catalog}`.brz_market.raw_coinbase_ohlc_1d
            ADD CONSTRAINT ck_coinbase_ohlc_keys
            CHECK (product_id IS NOT NULL AND bar_date IS NOT NULL)
            """,
            "ck_coinbase_ohlc_prices": """
            ALTER TABLE `{catalog}`.brz_market.raw_coinbase_ohlc_1d
            ADD CONSTRAINT ck_coinbase_ohlc_prices
            CHECK (
                (open IS NULL OR open >= 0) AND
                (high IS NULL OR high >= 0) AND
                (low IS NULL OR low >= 0) AND
                (close IS NULL OR close >= 0) AND
                (volume IS NULL OR volume >= 0)
            )
            """,
        },
    },
    "brz_macro.raw_ecb_fx_ref_rates_daily": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_macro.raw_ecb_fx_ref_rates_daily (
            base_currency STRING NOT NULL COMMENT 'Typically EUR',
            quote_currency STRING NOT NULL COMMENT 'Quoted currency, e.g., USD',
            rate_date DATE NOT NULL COMMENT 'FX reference rate date',
            rate DOUBLE COMMENT 'ECB reference FX rate',
            ingested_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp (UTC)',
            run_id STRING NOT NULL COMMENT 'Pipeline run identifier',
            payload_hash STRING COMMENT 'Hash of raw source payload for audit/dedup'
        )
        USING DELTA
        COMMENT 'Bronze: raw ECB FX reference rates (daily). Logical PK = (base_currency, quote_currency, rate_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_ecb_fx_keys": """
            ALTER TABLE `{catalog}`.brz_macro.raw_ecb_fx_ref_rates_daily
            ADD CONSTRAINT ck_ecb_fx_keys
            CHECK (base_currency IS NOT NULL AND quote_currency IS NOT NULL AND rate_date IS NOT NULL)
            """,
            "ck_ecb_fx_rate": """
            ALTER TABLE `{catalog}`.brz_macro.raw_ecb_fx_ref_rates_daily
            ADD CONSTRAINT ck_ecb_fx_rate
            CHECK (rate IS NULL OR rate > 0)
            """,
        },
    },
    "brz_macro.raw_fred_series": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_macro.raw_fred_series (
            series_id STRING NOT NULL COMMENT 'FRED series id, e.g., CPIAUCSL',
            observation_date DATE NOT NULL COMMENT 'Observation date',
            realtime_start DATE NOT NULL COMMENT 'Revision window start (FRED realtime)',
            realtime_end DATE NOT NULL COMMENT 'Revision window end (FRED realtime)',
            value DOUBLE COMMENT 'Numeric observation value',
            ingested_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp (UTC)',
            run_id STRING NOT NULL COMMENT 'Pipeline run identifier',
            payload_hash STRING COMMENT 'Hash of raw payload for audit/dedup'
        )
        USING DELTA
        COMMENT 'Bronze: raw FRED observations (long table). Logical PK = (series_id, observation_date, realtime_start, realtime_end).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_fred_keys": """
            ALTER TABLE `{catalog}`.brz_macro.raw_fred_series
            ADD CONSTRAINT ck_fred_keys
            CHECK (
                series_id IS NOT NULL AND observation_date IS NOT NULL AND
                realtime_start IS NOT NULL AND realtime_end IS NOT NULL
            )
            """,
            "ck_fred_realtime_order": """
            ALTER TABLE `{catalog}`.brz_macro.raw_fred_series
            ADD CONSTRAINT ck_fred_realtime_order
            CHECK (realtime_start <= realtime_end)
            """,
        },
    },
}

other_table_ddls = {
    "slv_market.crypto_ohlc_1d": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_market.crypto_ohlc_1d (
            product_id STRING NOT NULL COMMENT 'Logical PK part 1',
            bar_date DATE NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING COMMENT 'Parsed from product_id',
            quote_currency STRING COMMENT 'Parsed from product_id',
            open DOUBLE,
            high DOUBLE,
            low DOUBLE,
            close DOUBLE,
            volume DOUBLE,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Silver: cleaned daily crypto OHLC bars. Logical PK = (product_id, bar_date).'
    """,
    "gld_market.dp_crypto_returns_1d": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_market.dp_crypto_returns_1d (
            product_id STRING NOT NULL COMMENT 'Logical PK part 1',
            bar_date DATE NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING,
            quote_currency STRING,
            close DOUBLE,
            simple_return_1d DOUBLE,
            log_return_1d DOUBLE,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold market product: daily crypto returns. Logical PK = (product_id, bar_date).'
    """,
    "gld_market.dp_crypto_volatility_1d": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_market.dp_crypto_volatility_1d (
            product_id STRING NOT NULL COMMENT 'Logical PK part 1',
            bar_date DATE NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING,
            quote_currency STRING,
            simple_return_1d DOUBLE,
            volatility_7d DOUBLE,
            volatility_30d DOUBLE,
            volatility_90d DOUBLE,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold market product: daily realized volatility. Logical PK = (product_id, bar_date).'
    """,
    "slv_macro.ecb_fx_ref_rates_daily": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_macro.ecb_fx_ref_rates_daily (
            base_currency STRING NOT NULL COMMENT 'Logical PK part 1',
            quote_currency STRING NOT NULL COMMENT 'Logical PK part 2',
            rate_date DATE NOT NULL COMMENT 'Logical PK part 3',
            rate DOUBLE,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Silver: cleaned ECB FX daily rates. Logical PK = (base_currency, quote_currency, rate_date).'
    """,
    "slv_macro.fred_series_clean": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_macro.fred_series_clean (
            series_id STRING NOT NULL COMMENT 'Logical PK part 1',
            observation_date DATE NOT NULL COMMENT 'Logical PK part 2',
            realtime_start DATE NOT NULL COMMENT 'Logical PK part 3',
            realtime_end DATE NOT NULL COMMENT 'Logical PK part 4',
            value DOUBLE,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Silver: cleaned FRED series without resampling. Logical PK = (series_id, observation_date, realtime_start, realtime_end).'
    """,
    "gld_macro.dp_macro_indicators": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_macro.dp_macro_indicators (
            indicator_id STRING NOT NULL COMMENT 'Logical PK part 1',
            observation_date DATE NOT NULL COMMENT 'Logical PK part 2',
            value DOUBLE,
            source_system STRING NOT NULL,
            indicator_group STRING,
            frequency STRING,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold macro product stored as a long table. Logical PK = (indicator_id, observation_date).'
    """,
    "gld_cross.dp_crypto_macro_features_1d": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_cross.dp_crypto_macro_features_1d (
            feature_date DATE NOT NULL COMMENT 'Logical PK part 1',
            product_id STRING NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING,
            quote_currency STRING,
            simple_return_1d DOUBLE,
            log_return_1d DOUBLE,
            volatility_7d DOUBLE,
            volatility_30d DOUBLE,
            macro_features MAP<STRING, DOUBLE>,
            fill_policy STRING,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold cross-domain feature table. Logical PK = (feature_date, product_id).'
    """,
    "obs.obs_ingestion_state": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.obs.obs_ingestion_state (
            pipeline_name STRING,
            source_name STRING,
            target_table STRING,
            watermark_value STRING,
            watermark_type STRING,
            last_success_at TIMESTAMP,
            last_run_id STRING,
            status STRING,
            updated_at TIMESTAMP
        )
        USING DELTA
        COMMENT 'Latest ingestion checkpoint state per pipeline.'
    """,
    "obs.obs_pipeline_run_log": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.obs.obs_pipeline_run_log (
            run_id STRING,
            pipeline_name STRING,
            layer STRING,
            source_name STRING,
            target_table STRING,
            status STRING,
            rows_read BIGINT,
            rows_written BIGINT,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING,
            metadata_json STRING
        )
        USING DELTA
        COMMENT 'Pipeline execution log for observability and troubleshooting.'
    """,
}

for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{schema}")

bronze_tables_ensured = []
bronze_constraints_added = []
bronze_constraints_existing = []

for table_name, spec in bronze_table_specs.items():
    full_name = f"{catalog}.{table_name}"
    spark.sql(spec["ddl"].format(catalog=catalog))
    bronze_tables_ensured.append(full_name)

    for constraint_name, constraint_sql in spec["constraints"].items():
        if constraint_exists(full_name, constraint_name):
            bronze_constraints_existing.append(f"{full_name}:{constraint_name}")
            continue

        spark.sql(constraint_sql.format(catalog=catalog))
        bronze_constraints_added.append(f"{full_name}:{constraint_name}")

for ddl in other_table_ddls.values():
    spark.sql(ddl.format(catalog=catalog))

result = {
    "status": "success",
    "catalog": catalog,
    "schemas_created": [f"{catalog}.{schema}" for schema in schemas],
    "bronze_tables_ensured": bronze_tables_ensured,
    "bronze_constraints_added": bronze_constraints_added,
    "bronze_constraints_existing": bronze_constraints_existing,
    "other_tables_ensured": [f"{catalog}.{table_name}" for table_name in other_table_ddls],
    "table_count": len(bronze_table_specs) + len(other_table_ddls),
}

result